In [ ]:
# ==========================================
# CELL: TEST SET EVALUATION (RMSE & Correlation)
# ==========================================
import torch
import numpy as np
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# 1. Device and Mixed Precision Setup (For 4GB GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running evaluation on: {device}")

# CRITICAL: Re-initialize the test_loader so it yields the correct labels for RMSE calculation.
# Make sure your test dataset actually has the Valence/Arousal scores included!
test_loader_with_labels = DataLoader(
    CDSA_Dataset(test_samples, tokenizer, is_test=False), # False ensures we load the true scores
    batch_size=8, # Keep batch size small for the 4GB GPU
    shuffle=False
)

# 2. Load the Model Architecture and Weights
eval_model = CDSA_HybridModel().to(device)
eval_model.load_state_dict(torch.load("best_hybrid_model.pt", map_location=device))

# Turn off learning layers (like Dropout)
eval_model.eval()

all_preds = []
all_labels = []

# 3. Evaluation Loop
print("Starting evaluation...")
with torch.no_grad(): # Disable gradient calculation to save massive amounts of VRAM
    for batch in test_loader_with_labels:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Keep true labels on CPU for Sklearn math later
        true_labels = batch["labels"].numpy()

        # Use mixed precision for inference too (runs faster and uses less memory)
        with torch.cuda.amp.autocast():
            preds, _ = eval_model(input_ids, attention_mask)

        # Clamp predictions to your 1.0 - 9.0 SemEval scale
        preds = torch.clamp(preds, min=1.0, max=9.0)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(true_labels)

# 4. Stack arrays for final math
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

# 5. Calculate Final Metrics for the Report
v_rmse = np.sqrt(mean_squared_error(all_labels[:, 0], all_preds[:, 0]))
a_rmse = np.sqrt(mean_squared_error(all_labels[:, 1], all_preds[:, 1]))

v_pearson, _ = pearsonr(all_labels[:, 0], all_preds[:, 0])
a_pearson, _ = pearsonr(all_labels[:, 1], all_preds[:, 1])

# 6. Print the Table-Ready Results
print("\n" + "=" * 45)
print(" FINAL CROSS-DOMAIN TEST SET RESULTS")
print("=" * 45)
print(f" Valence RMSE:        {v_rmse:.4f}")
print(f" Arousal RMSE:        {a_rmse:.4f}")
print("-" * 45)
print(f" Valence Correlation: {v_pearson:.4f}")
print(f" Arousal Correlation: {a_pearson:.4f}")
print("=" * 45)